In [4]:
# BattingEdge Audit 2.0
# Run this notebook from inside BattingEdge_FYP/notebooks
# If you launch Jupyter from notebooks/, ROOT must point one level up

from pathlib import Path
import os
import numpy as np

# Adjust this path if needed
ROOT = Path("..")   # project root: BattingEdge_FYP

VIDEO_EXTS = [".mp4", ".avi", ".mov"]
FEATURE_EXT = ".npy"
MODEL_EXTS = [".keras", ".pkl", ".pt", ".npy"]
NOTEBOOK_EXT = ".ipynb"

def scan_files(root: Path, exts):
    return [f for f in root.rglob("*") if f.suffix in exts]

def count_by_class(folder: Path, exts):
    counts = {}
    if not folder.exists(): return counts
    for c in sorted(folder.glob("*")):
        if not c.is_dir(): continue
        files = [f for f in c.rglob("*") if f.suffix in exts]
        counts[c.name] = len(files)
    return counts

print("=== BattingEdge Audit 2.0 ===")

# 1. Notebooks
nb_root = ROOT / "notebooks"
notebooks = scan_files(nb_root, [NOTEBOOK_EXT])
print(f"\n📒 Found {len(notebooks)} notebooks in {nb_root}:\n")

categories = {"debug": [], "utility": [], "feature": [], "training": [], "inference": [], "other": []}
for nb in notebooks:
    name = nb.name.lower()
    if "debug" in name:
        categories["debug"].append(nb)
    elif "utility" in name or "csv" in name or "organize" in name:
        categories["utility"].append(nb)
    elif "feature" in name:
        categories["feature"].append(nb)
    elif "train" in name or "model" in name:
        categories["training"].append(nb)
    elif "infer" in name or "pose" in name:
        categories["inference"].append(nb)
    else:
        categories["other"].append(nb)

for cat, files in categories.items():
    print(f"🔹 {cat.upper()} ({len(files)}):")
    for f in files:
        print("   -", f.name)

# 2. Models
models = scan_files(ROOT / "backend" / "models", MODEL_EXTS)
print(f"\n🧠 Found {len(models)} model/scaler/encoder files in /backend/models:\n")
for f in sorted(models):
    print(" -", f.name)

# 3. Videos
video_root = ROOT / "data"
video_sets = ["dataset_v7_clean", "dataset_v8_balanced_videos"]
for ds in video_sets:
    ds_path = video_root / ds
    if not ds_path.exists(): continue
    print(f"\n🎥 Video dataset: {ds}")
    for split in ["train", "validation", "test"]:
        split_path = ds_path / split
        if not split_path.exists(): continue
        counts = count_by_class(split_path, VIDEO_EXTS)
        total = sum(counts.values())
        print(f"  {split.upper()} ({total} videos):")
        for cls, n in counts.items():
            print(f"    - {cls}: {n}")

# 4. Features
feature_root = ROOT / "data" / "features"
feature_sets = ["dataset_v7_99feat", "dataset_v8p"]
for fs in feature_sets:
    fs_path = feature_root / fs
    if not fs_path.exists(): continue
    print(f"\n📊 Feature set: {fs}")
    for split in ["train", "validation", "test"]:
        split_path = fs_path / split
        if not split_path.exists(): continue
        counts = count_by_class(split_path, [FEATURE_EXT])
        total = sum(counts.values())
        print(f"  {split.upper()} ({total} feature files):")
        for cls, n in counts.items():
            print(f"    - {cls}: {n}")

# 5. Orphans
print("\n🧹 Checking for orphaned or misnamed files:")

orphans = [f for f in ROOT.rglob("*.npy") if "features" not in str(f)]
if orphans:
    print(f"Found {len(orphans)} .npy files outside /features:")
    for f in orphans:
        print(" -", f.relative_to(ROOT))
else:
    print("No orphaned .npy files found.")

vid_orphans = [f for f in ROOT.rglob("*") if f.suffix in VIDEO_EXTS and "dataset" not in str(f)]
if vid_orphans:
    print(f"\nFound {len(vid_orphans)} video files outside /dataset folders:")
    for f in vid_orphans:
        print(" -", f.relative_to(ROOT))
else:
    print("No orphaned video files found.")

# 6. Summary
print("\n✅ Audit complete. Summary:")
print(f" - Notebooks: {len(notebooks)}")
print(f" - Models: {len(models)}")
print(f" - Video sets scanned: {video_sets}")
print(f" - Feature sets scanned: {feature_sets}")


=== BattingEdge Audit 2.0 ===

📒 Found 28 notebooks in ..\notebooks:

🔹 DEBUG (3):
   - 0_Debug_Paths.ipynb
   - 0_Debug_Shape.ipynb
   - 0_Debug_V7_Crash.ipynb
🔹 UTILITY (9):
   - 0_Utility_Count_Videos.ipynb
   - 0_Utility_CSV.ipynb
   - 0_Utility_Data_Scanner.ipynb
   - 0_Utility_Generate_Master_CSV_Clean.ipynb
   - 0_Utility_Generate_Unified_CSV.ipynb
   - 0_Utility_Organize_Dataset.ipynb
   - 0_Utility_Organize_Dataset_V8_Balance.ipynb
   - 0_Utility_Renamer.ipynb
   - 0_Utility_Verify_Features.ipynb
🔹 FEATURE (4):
   - 2D_Feature_Engineering_HF_Local.ipynb
   - 2E_Feature_Engineering_V7.ipynb
   - 2F_Feature_Engineering_V7_Clean.ipynb
   - 2_Feature_Engineering.ipynb
🔹 TRAINING (9):
   - 3_Model_Training_Error.ipynb
   - 3_Model_Training_V3_RF_FAIL.ipynb.ipynb
   - 3_Model_Training_V4_LSTM_FAIL.ipynb.ipynb
   - 3_Model_Training_V5-FIXED.ipynb
   - 3_Model_Training_V5_Shot_Classifier.ipynb
   - 3_Model_Training_V6_FineTune_FAIL.ipynb.ipynb
   - 3_Model_Training_V7_Diagnostic.ipynb